In [2]:
import time
import math

import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

I0000 00:00:1795173379.377442   51538 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795173379.379144   51538 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1795173379.684251   51538 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795173385.375187   51538 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

In [16]:
MODEL_PATH = "face_landmarker.task"
CAMERA_ID = 0

IRIS_L = 468
IRIS_R = 473
NOSE_TIP = 1

BEAM_OPENING_DEG = 20
BEAM_LENGTH = 4
BEAM_COLOR = (0, 200, 255)  # BGR
BEAM_THICKNESS = 3

YAW_GAIN = 2.5
PITCH_GAIN = 2.5
ANGLE_LIMIT = 0.5
SMOOTH = 0.3

In [4]:
def create_detector():
    base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
    options =  vision.FaceLandmarkerOptions(
        base_options = base_options,
        running_mode = vision.RunningMode.VIDEO,
        num_faces = 1
    )
    return vision.FaceLandmarker.create_from_options(options)

In [10]:
def get_face_points(result, w, h):
    if not result.face_landmarks:
        return None

    points = result.face_landmarks[0]

    left = points[IRIS_L]
    right = points[IRIS_R]
    nose = points[NOSE_TIP]
    
    if right.x < left.x:
        left, right = right, left 

    left_px = (left.x * w, left.y * h)
    right_px = (right.x * w, right.y * h)
    nose_px = (nose.x * w, nose.y * h)

    return left_px, right_px, nose_px


In [6]:
def to_int(p):
    return(int(round(p[0])), int(round(p[1])))

In [7]:
def beam_end(eye, angle, length):
    return (
        eye[0] + length * math.cos(angle),
        eye[1] + length * math.sin(angle)
    )

In [11]:
def clamp(x, lo, hi):
    return max(lo, min(hi, x))

In [12]:
def face_to_img_angle(dx, dy, theta):
    u = (math.cos(theta), math.sin(theta))
    v = (-math.sin(theta), math.cos(theta))
    ix = dx * u[0] + dy * v[0]
    iy = dx * u[1] + dy * v[1]
    return math.atan2(iy, ix)

In [ ]:
detector = create_detector()

v = cv2.VideoCapture(CAMERA_ID)

t_0 = time.perf_counter()

smooth_yaw = None
smooth_pitch = None
yaw_0 = 0.0
pitch_0 = 0.0

calibrated = False

while True:
    ret, frame = v.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    h, w = frame.shape[:2]

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

    timestamp = int((time.perf_counter() - t_0) * 1000)
    r = detector.detect_for_video(mp_img, timestamp)

    face = get_face_points(r, w, h)

    face_ok = face is not None
    if face_ok:
        eye_l, eye_r, nose = face
        
        eye_v = (eye_r[0] - eye_l[0], eye_r[1] - eye_l[1])
        eye_mod = math.hypot(eye_v[0], eye_v[1])
        theta = math.atan2(eye_v[1], eye_v[0])

        u = (math.cos(theta), math.sin(theta))
        v_axis = (-math.sin(theta), math.cos(theta))

        mid = ((eye_l[0] + eye_r[0])/2, (eye_l[1]+eye_r[1])/2)
        nose_v = (nose[0] - mid[0], nose[1] - mid[1])
        yaw = (nose_v[0] * u[0] + nose_v[1] * u[1]) / eye_mod
        pitch = (nose_v[0] * v_axis[0] + nose_v[1] * v_axis[1]) / eye_mod

        if smooth_yaw is None:
            smooth_yaw, smooth_pitch = yaw, pitch
        else:
            smooth_yaw += SMOOTH * (yaw - smooth_yaw)
            smooth_pitch += SMOOTH * (pitch - smooth_pitch)

        yaw = clamp(smooth_yaw - yaw_0, -ANGLE_LIMIT, ANGLE_LIMIT)
        pitch = clamp(smooth_pitch - pitch_0, -ANGLE_LIMIT, ANGLE_LIMIT)

        phi = math.radians(BEAM_OPENING_DEG)
        shift_x = YAW_GAIN * yaw
        shift_y = PITCH_GAIN * pitch

        dir_l = (-math.sin(phi) + shift_x, math.cos(phi) + shift_y)
        dir_r = (math.sin(phi) + shift_x, math.cos(phi) + shift_y)
 
        angle_l = face_to_img_angle(dir_l[0], dir_l[1], theta)
        angle_r = face_to_img_angle(dir_r[0], dir_r[1], theta)

        length = BEAM_LENGTH * eye_mod

        end_l = beam_end(eye_l, angle_l, length)
        end_r = beam_end(eye_r, angle_r, length)

        cv2.line(
             frame, to_int(eye_l), to_int(end_l),
             BEAM_COLOR, BEAM_THICKNESS, cv2.LINE_AA
        )
        cv2.line(
             frame, to_int(eye_r), to_int(end_r),
             BEAM_COLOR, BEAM_THICKNESS, cv2.LINE_AA
        )

        if not calibrated:
            cv2.putText(
                frame, "Mira de frente y pulsa C para calibrar", (20, h - 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 200, 255), 2
            )

        """
        for center in eyes:
            cv2.circle(frame, center, 6, (0, 255, 255), 2)
            cv2.circle(frame, center, 1, (0, 0, 255), -1)

        cv2.line(frame, eyes[0], eyes[1], (0, 255, 0), 1)
        """
    else:
        smooth_yaw = None
        smooth_pitch = None
        
        cv2.putText(
            frame, "Sin cara", (20, 60),
            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2
        )

    cv2.imshow("Filtro", frame)

    key = cv2.waitKey(1) & 0xFF
    if key in (ord("q"), 27):
            break
    elif key == ord("c") and face_ok:
        # Calibrar
        yaw_0 = smooth_yaw
        pitch_0 = smooth_pitch
        calibrated = True
    
v.release()
cv2.destroyAllWindows()
detector.close()

W0000 00:00:1795178563.832806   56315 face_landmarker_graph.cc:181] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
W0000 00:00:1795178563.848817   56317 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1795178563.890607   56317 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
